In [3]:
import os
import time
import requests
import numpy as np
import pandas as pd

HUBS = [
    ("Kafr_ElSheikh", 31.11, 30.94), ("Dakahlia", 31.04, 31.38),
    ("Sharqia", 30.72, 31.67), ("Beheira", 31.03, 30.47),
    ("Gharbia", 30.79, 31.00), ("Menoufia", 30.55, 31.01),
    ("Qalyubia", 30.46, 31.18), ("Salhia_Reclaimed", 30.65, 31.95),
    ("Fayoum", 29.31, 30.84), ("Beni_Suef", 29.07, 31.09),
    ("Minya", 28.09, 30.75), ("Assiut", 27.18, 31.18),
    ("Sohag", 26.56, 31.69), ("Qena_Luxor", 25.69, 32.64),
]

CHUNKS = [
    ("2016-01-01", "2017-12-31"),
    ("2018-01-01", "2019-12-31"),
    ("2020-01-01", "2021-12-31"),
    ("2022-01-01", "2023-12-31"),
    ("2024-01-01", "2025-12-31"),
]

date_bins = pd.date_range("2016-01-01", "2025-12-31", freq="10D")
expected_dates = pd.to_datetime(date_bins[:-1])

def fetch_era5_chunk(lat, lon, start_d, end_d, retries=6):
    url = (
        "https://archive-api.open-meteo.com/v1/archive?"
        f"latitude={lat}&longitude={lon}&start_date={start_d}&end_date={end_d}"
        "&daily=et0_fao_evapotranspiration,vapour_pressure_deficit_max"
        "&hourly=soil_moisture_0_to_7cm,soil_moisture_7_to_28cm,soil_moisture_28_to_100cm,soil_temperature_0_to_7cm"
        "&timezone=Africa%2FCairo"
    )
    for _ in range(retries):
        try:
            r = requests.get(url, timeout=60)
            js = r.json()
            if r.status_code == 200 and "daily" in js and "hourly" in js:
                return js
            print("      ⏳ انتظار 65 ثانية لتجدد حد الطلبات في Open-Meteo...", flush=True)
            time.sleep(65)
        except Exception:
            time.sleep(10)
    raise RuntimeError(f"فشل سحب ERA5 للفترة {start_d} - {end_d}")

dfs = []
print("📥 [2/6] بدء سحب بيانات ERA5-Land لـ 14 محافظة...", flush=True)

for name, lat, lon in HUBS:
    parts = []
    for s_d, e_d in CHUNKS:
        res = fetch_era5_chunk(lat, lon, s_d, e_d)
        df_d = pd.DataFrame(res["daily"]).rename(columns={
            "time": "Date",
            "et0_fao_evapotranspiration": "ERA5_ET0_mm",
            "vapour_pressure_deficit_max": "ERA5_VPD_Max_kPa",
        })
        df_d["Date"] = pd.to_datetime(df_d["Date"])

        df_h = pd.DataFrame(res["hourly"])
        df_h["Date"] = pd.to_datetime(df_h["time"]).dt.floor("D")
        df_s = df_h.groupby("Date", as_index=False)[
            ["soil_moisture_0_to_7cm", "soil_moisture_7_to_28cm", "soil_moisture_28_to_100cm", "soil_temperature_0_to_7cm"]
        ].mean().rename(columns={
            "soil_moisture_0_to_7cm": "ERA5_SoilMoi_0_7cm_cm3",
            "soil_moisture_7_to_28cm": "ERA5_SoilMoi_7_28cm_cm3",
            "soil_moisture_28_to_100cm": "ERA5_SoilMoi_28_100cm_cm3",
            "soil_temperature_0_to_7cm": "ERA5_SoilTemp_0_7cm_C",
        })
        parts.append(pd.merge(df_d, df_s, on="Date", how="outer"))
        time.sleep(1.5)

    df_gov = pd.concat(parts, ignore_index=True).drop_duplicates(subset=["Date"]).sort_values("Date")
    df_gov["Bin"] = pd.cut(df_gov["Date"], bins=date_bins, right=False, labels=expected_dates)
    df_10d = df_gov.groupby("Bin", observed=True).mean(numeric_only=True).reindex(expected_dates).reset_index()
    df_10d.rename(columns={"index": "Date"}, inplace=True)
    df_10d["Governorate"] = name
    dfs.append(df_10d)
    print(f"   ✓ {name}", flush=True)

out = pd.concat(dfs, ignore_index=True).sort_values(["Governorate", "Date"]).reset_index(drop=True)
num_cols = out.select_dtypes(include=[np.number]).columns
out[num_cols] = out.groupby("Governorate")[num_cols].transform(lambda g: g.interpolate("linear").ffill().bfill())
out["Date"] = out["Date"].dt.strftime("%Y-%m-%d")

cols = ["Governorate", "Date"] + [c for c in out.columns if c not in ["Governorate", "Date"]]
out = out[cols]

output_path = r"d:\vscode\NASA\DATA\02_ERA5_Land_Hydrology.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
out.to_csv(output_path, index=False)
print(f"✅ اكتمل الملف الثاني: {output_path} | الأبعاد: {out.shape}\n")

📥 [2/6] بدء سحب بيانات ERA5-Land لـ 14 محافظة...


   ✓ Kafr_ElSheikh
   ✓ Dakahlia
   ✓ Sharqia
   ✓ Beheira
      ⏳ انتظار 65 ثانية لتجدد حد الطلبات في Open-Meteo...
   ✓ Gharbia
   ✓ Menoufia
   ✓ Qalyubia
   ✓ Salhia_Reclaimed
      ⏳ انتظار 65 ثانية لتجدد حد الطلبات في Open-Meteo...
   ✓ Fayoum
   ✓ Beni_Suef
   ✓ Minya
   ✓ Assiut
   ✓ Sohag
   ✓ Qena_Luxor
✅ اكتمل الملف الثاني: d:\vscode\NASA\DATA\02_ERA5_Land_Hydrology.csv | الأبعاد: (5110, 8)

